# 06. 고급 RAG — HyDE / RAG-Fusion

**무엇을 하나:** 쿼리를 그대로 임베딩하지 않고 LLM 으로 보강해 검색 품질을 높인다.
- **HyDE**(짧은 쿼리용): 쿼리로 *가상의 레시피 문서*를 LLM이 써서, 그걸 임베딩해 검색 — 질문과 문서의 표현 간극을 줄인다.
- **RAG-Fusion**(복합 쿼리용): 관점이 다른 여러 쿼리를 LLM이 만들어 각각 검색한 뒤 **RRF**(Reciprocal Rank Fusion)로 순위를 합산한다.
- 라우팅: 공백 제거 길이 `< 10` → HyDE / `>= 10` → RAG-Fusion (`choose_rag_strategy`).

**모델 티어 (중요):** 이 서브쿼리 LLM = **LM Studio 로컬 qwen2.5-7b**(`RAG_LLM_PROVIDER=local`). 검색 보조라 작은 모델로 충분하다(오류 허용도 큼). 큰 오케스트레이션(의도해석·식단생성)은 Claude — 상세 `docs/agent/llm-models.md`.

**폴백 확인:** 서브 LLM 이 안 닿으면 **조용히 plain 벡터검색으로 폴백**한다. 아래 `_via` 가 `hyde`/`rag_fusion` 이면 실발동, `plain폴백` 이면 서브LLM 미동작(LM Studio/키 확인).

In [ ]:
import sys,os,asyncio
from pathlib import Path
B=(Path.cwd().parent/'backend') if Path.cwd().name=='notebooks' else Path.cwd()/'backend'
B=B.resolve(); sys.path.insert(0,str(B)); os.chdir(B)
try: sys.stdout.reconfigure(encoding='utf-8')   # Windows 콘솔/nbconvert cp949 방어
except Exception: pass
from dotenv import load_dotenv; load_dotenv()
from app.rag.hyde import hyde_search
from app.rag.rag_fusion import rag_fusion_search
from app.rag._llm import rag_subquery_provider, llm_enabled
print('서브쿼리 provider:', rag_subquery_provider(), '| enabled:', llm_enabled(rag_subquery_provider()))

def show(tag, docs):
    print(tag)
    for d in docs:
        via = d.get('_via', 'plain폴백')   # _via 없으면 LLM 실패->plain 폴백된 것
        score = d.get('_rrf_score') if d.get('_rrf_score') is not None else round(d.get('distance', 0), 3)
        print(f'   {d.get("name"):<20} via={via}  rrf/dist={score}')

show('HyDE  된장찌개:', asyncio.run(hyde_search('된장찌개', k=3)))
show('Fusion 매콤한 돼지고기 볶음 한식:', asyncio.run(rag_fusion_search('매콤한 돼지고기 볶음 한식', k=3)))
print('* via=hyde/rag_fusion => 실발동 / via=plain폴백 => 서브LLM 미동작(LM Studio/키 확인)')